# Cvrie
Project :
Train a model to sort XRAY images base on the body part of the image.

### Fix project root for imports

In [ ]:
import sys
from pathlib import Path

projectRoot = Path.cwd().parent
if str(projectRoot) not in sys.path:
    sys.path.append(str(projectRoot))

## Load Images Dataset

### Dataset Justification
We choosed multiple image dataset, all images are XRAY scans of a human body part.

Our objectif is to train a model to categorize an XRAY image by body parts.

### Defining body parts list

In [ ]:
bodyPartsList = [
    "Chest"
    ,"Head"
    ,"Hand"
    ,"Foot"
]

print("Working body parts:", bodyPartsList)


### Normalization pipeline

```mermaid
graph LR
    classDef input    fill:#1E88E5,stroke:#1565C0,color:#fff
    classDef process  fill:#E53935,stroke:#B71C1C,color:#fff
    classDef storage  fill:#FB8C00,stroke:#EF6C00,color:#fff
    classDef output   fill:#43A047,stroke:#2E7D32,color:#fff

    A([🖼️ Hand.jpg]):::input
    A --> B([Conversion Grayscale]):::process
    B --> C([Size Normalization]):::process
    C -->|Insertion| D([💾 /Dataset/*/Normalized]):::storage

    D -->|Fetch| E([Min-Max Normalization]):::process
    E --> F([🔢 Pixel Array flatten]):::output
```

In [ ]:
from Source.imageNormalizer import imageNormalizer

print("Normalizing images...")
imageNormalizer(bodyPartsList)
print("Done normalizing images.")

### Create dataset by body parts
For each image bank, a dataset with ImageId, Label(bodyPart) will be created

In [ ]:
from Source.createDataset import prepareDataset

createdCsvPaths = []

print("Creating datasets for body parts...")

for bodyPart in bodyPartsList:
    csvPath = prepareDataset(bodyPart=bodyPart)
    createdCsvPaths.append(csvPath)
    print(f"Dataset for {bodyPart} created at {csvPath}")

### Merge all body parts dataset to one big dataset

In [ ]:
from Source.mergeDataset import mergeDataset

print("Merging body parts datasets...")

mergedDataset = mergeDataset(csvPaths=createdCsvPaths, outputPath=projectRoot / "Dataset" / "mergedDataset.csv")
print(f"Merged dataset rows: {len(mergedDataset)}")

## Analyse Data

### Load mergedDataset

In [ ]:
import pandas as pd

df = pd.read_csv(projectRoot / "Dataset" / "mergedDataset.csv")

### Check for missing values or duplicate rows

In [ ]:
print(f"Dataset size : {df.shape[0]} rows / {df.shape[1]} columns")

print("\nMissing values :\n", df.isnull().sum())
print(f"\nNumber of duplicate rows : {df.duplicated().sum()}")

display(df.head())

### Check for body parts repartition

In [ ]:
import matplotlib.pyplot as plt

# Class distribution counts
classCounts = df['Label(BodyPart)'].value_counts()
display(classCounts)

# Bar chart for class distribution
plt.figure(figsize=(10, 5))
plt.title("Dataset Distribution by Body Part (Classes)", fontsize=14)
plt.bar(x=classCounts.index, height=list(classCounts.values))
plt.xlabel("Body Part")
plt.ylabel("Number of Images")
plt.grid(axis='y', linestyle='--')
plt.show()

# Pie chart for class distribution
plt.figure(figsize=(5, 5))
plt.title("Proportion of Classes in the Dataset")
plt.pie(classCounts, labels=list(classCounts.index), autopct='%1.1f%%')
plt.show()

### Check normalization result

In [ ]:
import numpy as np
from Source.imageLoader import loadImage

# Expected image size: 64x64
imgSize = int(np.sqrt(4096))

plt.figure(figsize=(16, 4))

for i, bodyPart in enumerate(bodyPartsList):
    sampleImgId = df[df['Label(BodyPart)'] == bodyPart].iloc[0]['ImageId']
    normalizedImage = loadImage(sampleImgId, bodyPart)
    
    img2d = np.array(normalizedImage).reshape(imgSize, imgSize)
    
    plt.subplot(1, len(bodyPartsList), i + 1)
    plt.imshow(img2d, cmap='gray')
    plt.title(f"{bodyPart}\nID: {sampleImgId}")
    plt.axis('off')

plt.tight_layout()
plt.show()

## Model training

**Handling Imbalanced Data :**

**stratify=y in train_test_split**: preserves the same class proportions in the training and test sets.

**class_weight='balanced' in models**: gives more weight to minority classes so the model pays more attention to underrepresented categories.

### Preparing features and labels

In [ ]:
import numpy as np
import pandas as pd
from Source.imageLoader import loadImage

df = pd.DataFrame(mergedDataset)

# X = features, y = labels
print("Preparing features (X) and labels (y)...")
XList = []
for index, row in df.iterrows():
    normalizedImage = loadImage(row['ImageId'], row['Label(BodyPart)'])
    XList.append(normalizedImage)

X = np.array(XList)
y = df['Label(BodyPart)'].values

print(f"Features Format (X) : {X.shape}")

### Random Forest

#### RF: Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

print("Random Forest Classifier Training...")

# Train model
print("Data splitting (80% Train / 20% Test)...")
yStratifyRF = list(y)
XTrainRF, XTestRF, yTrainRF, yTestRF = train_test_split(X, y, test_size=0.2, random_state=42, stratify=yStratifyRF)

randomForest = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

print("Training...")
randomForest.fit(XTrainRF, yTrainRF)

joblib.dump(randomForest, projectRoot / "ModelSave/randomForest.pkl")

#### RF: Evaluating

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

print("Random Forest Evaluation...")

# Evaluate model
print("Evaluating...")
yPredRF = randomForest.predict(XTestRF)
accTrainRF = accuracy_score(yTrainRF, randomForest.predict(XTrainRF))
accTestRF = accuracy_score(yTestRF, yPredRF)

# Print results
print(f"\nTrain Accuracy: {accTrainRF * 100:.2f}%")
print(f"Test Accuracy: {accTestRF * 100:.2f}%")
if accTrainRF - accTestRF > 0.1:
    print("Warning: Potential overfitting.")
else:
    print("No overfitting detected.")
print("Classification report RF :")
print(classification_report(yTestRF, yPredRF))

# Confusion Matrix
# fig -> Figure object
# ax -> Axes object
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(yTestRF, yPredRF, ax=ax, cmap='Blues')
plt.title("Confusion Matrix - Random Forest")
plt.show()

#### RF: Finding best configuration

Random Forest optimisation : We search for the best configuration to have the most precise RF model using a GridSearch.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
import joblib

print("Searching for the best hyperparameters for the Random Forest...")

# Create a parameter grid for GridSearchCV
paramGridRF = {
    'n_estimators': [100, 200, 300], # Numbers of trees in the forest
    'max_depth': [None, 10, 20],     # Maximum depth of the tree
    'min_samples_split': [2, 5, 10], # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4]    # Minimum number of samples required to be at a leaf node
}

# CV -> Cross Validation = 3 -> 3-fold cross-validation (dataset split into 3 parts, train on 2, test on 1, repeat 3 times)
# Scoring -> Metric evaluating the model (accuracy score)
# n_jobs=-1 -> Use all available CPU
# verbose=2 -> Print logs
gridSearchRF = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid=paramGridRF,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

# Training the GridSearchCV to find the best hyperparameters
gridSearchRF.fit(XTrainRF, yTrainRF)

print(f"\nBest parameters found : {gridSearchRF.best_params_}")
print(f"Best Score  : {gridSearchRF.best_score_:.4f}")

gridSearchResultsRF = pd.DataFrame(gridSearchRF.cv_results_)
gridSearchResultsRF = gridSearchResultsRF[['params', 'mean_test_score', 'std_test_score', 'mean_fit_time']]
gridSearchResultsRF = gridSearchResultsRF.sort_values('mean_test_score', ascending=False)
print(gridSearchResultsRF.to_string())

# Saves the best model found by GridSearchCV
bestRF = gridSearchRF.best_estimator_

joblib.dump(bestRF, projectRoot / "ModelSave/bestRandomForest.pkl")

#### RF: Evaluating best configuration

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
import matplotlib.pyplot as plt

print("Evaluating Best Random Forest...")

# Evaluate model
yPredBestRF = bestRF.predict(XTestRF)
accTrainBestRF = accuracy_score(yTrainRF, bestRF.predict(XTrainRF))
accTestBestRF = accuracy_score(yTestRF, yPredBestRF)

# Print results
print(f"Normal RF Train Accuracy : {accTrainRF * 100:.2f}%")
print(f"Normal RF Test Accuracy  : {accTestRF * 100:.2f}%")
print(f"Best RF Train Accuracy   : {accTrainBestRF * 100:.2f}%")
print(f"Best RF Test Accuracy    : {accTestBestRF * 100:.2f}%")
print("Classification report Best RF :")
print(classification_report(yTestRF, yPredBestRF))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(yTestRF, yPredBestRF, ax=ax, cmap='Blues')
plt.title("Matrice de Confusion - Random Forest Optimisé")
plt.show()


### Principal Component Analysis (PCA) and Support Vector Machine (SVM)

#### SVM: Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
import joblib

print("Support Vector Machine (SVM) Training...")

print("Data splitting (80% Train / 20% Test)...")
# Stratify -> Better class balance in train/test sets
yStratifyRF = list(y)
XTrainSVM, XTestSVM, yTrainSVM, yTestSVM = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=yStratifyRF
)

# Pipeline (Normalization -> Dimension Reduction -> Classification)
# SVC -> Support Vector Classifier
print("Building PCA + SVC Pipeline...")
pipelineSVC = Pipeline([
    # StandardScaler : Normalize data
    ('scaler', StandardScaler()), 

    # PCA : 95% seen data
    # 4096 pixels -> ~100-200 components
    ('pca', PCA(n_components=0.95, random_state=42)), 
    
    # SVM + RBF : Non-linear classification
    ('svm', SVC(kernel='rbf', class_weight='balanced', random_state=42)) 
])

# Train the pipeline
print("Training...")
pipelineSVC.fit(XTrainSVM, yTrainSVM)

joblib.dump(pipelineSVC, projectRoot / "ModelSave/pipelineSVC.pkl")

#### SVM: Evaluating

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import ConfusionMatrixDisplay

print("Support Vector Machine (SVM) Evaluation...")

# Evaluate model
print("Evaluating...")
yPredSVM = pipelineSVC.predict(XTestSVM)
accTrainSVM = accuracy_score(yTrainSVM, pipelineSVC.predict(XTrainSVM))
accTestSVM = accuracy_score(yTestSVM, yPredSVM)

# Display results
print(f"\nTrain Accuracy: {accTrainSVM * 100:.2f}%")
print(f"Test Accuracy: {accTestSVM * 100:.2f}%")
if accTrainSVM - accTestSVM > 0.1:
    print("Warning: Potential overfitting.")
else:
    print("No overfitting detected.")
print("Classification report SVM :")
print(classification_report(yTestSVM, yPredSVM))

# Show how many dimensions were kept after PCA
print("\nDimensions after PCA :")
n_components_kept = pipelineSVC.named_steps['pca'].n_components_
print(f"{XTrainSVM.shape[1]} pixels -> {n_components_kept} principal components.")

# Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(yTestSVM, yPredSVM, ax=ax, cmap='Blues')
plt.title("Confusion Matrix - PCA + SVM")
plt.show()

#### SVM: Finding best configuration

SVM optimisation : We search for the best configuration to have the most precise SVM model using a GridSearch.

In [ ]:
from sklearn.model_selection import GridSearchCV
import joblib
import pandas as pd

print("Searching for the best hyperparameters for the SVM...")

paramGridSVM = {
    "pca__n_components": [0.90, 0.95], # Keep 90% or 95% of variance
    "svm__C": [0.1, 1, 10],            # Regularization parameter (lower = more regularization)
    "svm__kernel": ["rbf", "poly"]     # RBF = non-linear, Poly = polynomial kernel
}

gridSearchSVM = GridSearchCV(
    estimator=pipelineSVC,
    param_grid=paramGridSVM,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

# Training the GridSearchCV to find the best hyperparameters
gridSearchSVM.fit(XTrainSVM, yTrainSVM)

print(f"\nBest parameters found : {gridSearchSVM.best_params_}")
print(f"Best Score  : {gridSearchSVM.best_score_:.4f}")

gridSearchResultsSVM = pd.DataFrame(gridSearchSVM.cv_results_)
gridSearchResultsSVM = gridSearchResultsSVM[['params', 'mean_test_score', 'std_test_score', 'mean_fit_time']]
gridSearchResultsSVM = gridSearchResultsSVM.sort_values('mean_test_score', ascending=False)
print(gridSearchResultsSVM.head().to_string())

# Saves the best model found by GridSearchCV
bestSVM = gridSearchSVM.best_estimator_

joblib.dump(bestSVM, projectRoot / "ModelSave/bestSVM.pkl")


#### SVM: Evaluating best configuration

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

print("Evaluating Best SVM...")

# Evaluate model
yPredBestSVM = bestSVM.predict(XTestSVM)
accTrainBestSVM = bestSVM.score(XTrainSVM, yTrainSVM)
accTestBestSVM = bestSVM.score(XTestSVM, yTestSVM)

# Print results
print(f"Normal SVM Train Accuracy : {accTrainSVM * 100:.2f}%")
print(f"Normal SVM Test Accuracy  : {accTestSVM * 100:.2f}%")
print(f"Best SVM Train Accuracy   : {accTrainBestSVM * 100:.2f}%")
print(f"Best SVM Test Accuracy    : {accTestBestSVM * 100:.2f}%")
print("Classification report Best SVM :")
print(classification_report(yTestSVM, yPredBestSVM))

# Show how many dimensions were kept after PCA
print("\nDimensions after PCA (Best Model) :")
n_components_kept_best = bestSVM.named_steps['pca'].n_components_
print(f"{XTrainSVM.shape[1]} pixels -> {n_components_kept_best} principal components.")

# Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(yTestSVM, yPredBestSVM, ax=ax, cmap='Blues')
plt.title("Confusion Matrix - PCA + SVM Optimisé")
plt.show()

### MLPclassifier model

#### MLP: Training

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import joblib

print("Multi-Layer Perceptron (MLP) Training...")

labelEncoder = LabelEncoder()
yEncoded = labelEncoder.fit_transform(list(y))

print("\nData splitting (80% Train / 20% Test)...")
XTrainMLP, XTestMLP, yTrainMLP, yTestMLP = train_test_split(X, yEncoded, test_size=0.2, random_state=42, stratify=yEncoded)

scaler = StandardScaler()
XTrainMLP = scaler.fit_transform(XTrainMLP)
XTestMLP = scaler.transform(XTestMLP)

modelMLP = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    random_state=42
)

print("Model training...")
modelMLP.fit(XTrainMLP, yTrainMLP)

joblib.dump(modelMLP, projectRoot / "ModelSave/modelMLP.pkl")
joblib.dump(scaler, projectRoot / "ModelSave/scaler.pkl")
joblib.dump(labelEncoder, projectRoot / "ModelSave/labelEncoder.pkl")

#### MLP: Evaluating

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay

print("Multi-Layer Perceptron (MLP) Evaluation...")

# Evaluate model
print("Evaluating...")
yPredMLP = modelMLP.predict(XTestMLP)
accTrainMLP = modelMLP.score(XTrainMLP, yTrainMLP) * 100
accTestMLP = modelMLP.score(XTestMLP, yTestMLP) * 100

# Display results
print(f"Train Accuracy  : {accTrainMLP:.2f}%")
print(f"Test Accuracy   : {accTestMLP:.2f}%")
if accTrainMLP - accTestMLP > 0.1:
    print("Warning: Potential overfitting.")
else:
    print("No overfitting detected.")
print("Classification report MLP :")
print(classification_report(yTestMLP, yPredMLP))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(yTestMLP, yPredMLP, ax=ax, cmap='Blues')
plt.title("Confusion Matrix - Multi-Layer Perceptron (MLP)")
plt.show()

#### MLP: Finding best configuration

MLP optimisation : We search for the best configuration to have the most precise MLP model using a GridSearch.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier
import joblib
import pandas as pd

print("Searching for the best hyperparameters for the MLP...")

paramGridMLP = {
    "hidden_layer_sizes": [(256, 256, 128), (256, 128, 64), (256, 128)],
    "learning_rate_init": [0.001],
    "activation": ["relu"],
    "random_state": [42]
}

gridSearchMLP = GridSearchCV(
    estimator=MLPClassifier(solver="adam", max_iter=300, early_stopping=True),
    param_grid=paramGridMLP,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

# Training the GridSearchCV to find the best hyperparameters
gridSearchMLP.fit(XTrainMLP, yTrainMLP)

print(f"\nBest parameters found : {gridSearchMLP.best_params_}")
print(f"Best Score  : {gridSearchMLP.best_score_:.4f}")

gridSearchResultsMLP = pd.DataFrame(gridSearchMLP.cv_results_)
gridSearchResultsMLP = gridSearchResultsMLP[['params', 'mean_test_score', 'std_test_score', 'mean_fit_time']]
gridSearchResultsMLP = gridSearchResultsMLP.sort_values('mean_test_score', ascending=False)
print(gridSearchResultsMLP.to_string())

# Saves the best model found by GridSearchCV
bestMLP = gridSearchMLP.best_estimator_

joblib.dump(bestMLP, projectRoot / "ModelSave/bestMLP.pkl")

#### MLP: Evaluating best configuration

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

print("Evaluating Best MLP...")

# Evaluate model
yPredBestMLP = bestMLP.predict(XTestMLP)
accTrainBestMLP = bestMLP.score(XTrainMLP, yTrainMLP) * 100
accTestBestMLP = bestMLP.score(XTestMLP, yTestMLP) * 100

# Print results
print(f"Normal MLP Train Accuracy : {accTrainMLP:.2f}%")
print(f"Normal MLP Test Accuracy  : {accTestMLP:.2f}%")
print(f"Best MLP Train Accuracy   : {accTrainBestMLP:.2f}%")
print(f"Best MLP Test Accuracy    : {accTestBestMLP:.2f}%")
print("Classification report Best MLP :")
print(classification_report(yTestMLP, yPredBestMLP))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(yTestMLP, yPredBestMLP, ax=ax, cmap='Blues')
plt.title("Confusion Matrix - Multi-Layer Perceptron (MLP) Optimisé")
plt.show()

# Loss curve
plt.figure()
plt.plot(bestMLP.loss_curve_)
plt.title("Loss curve — Best MLP")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

##### Loss Function Explanation

MLPClassifier : loss function used for multi-class classification is the **Log-Loss** (Cross-Entropy Loss).

**How it works:**

The Log-Loss evaluates the probability output of the model against the true label.

It heavily penalizes predictions that are confident but wrong.

At each iteration, our optimizer (`adam`) updates the network's weights to minimize this function.

As seen on the graph below, a decreasing loss curve = the model is successfully learning.

### Model Comparison

<table border="1" style="border-collapse: collapse; width: 100%;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="padding: 8px; text-align: left;">Model</th>
      <th style="padding: 8px; text-align: left;">Accuracy</th>
      <th style="padding: 8px; text-align: left;">Advantages</th>
      <th style="padding: 8px; text-align: left;">Disadvantages</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="padding: 8px;"><strong>Random Forest</strong></td>
      <td style="padding: 8px;">~98.77%</td>
      <td style="padding: 8px;">Robust to overfitting, fast training.</td>
      <td style="padding: 8px;">Considers pixels independently (loses spatial context).</td>
    </tr>
    <tr>
      <td style="padding: 8px;"><strong>Best Random Forest</strong></td>
      <td style="padding: 8px;">~98.97%</td>
    </tr>
      <td style="padding: 8px;"><strong>PCA + SVM</strong></td>
      <td style="padding: 8px;">~99.10%</td>
      <td style="padding: 8px;">Excellent with high-dimensional data, PCA reduces noise.</td>
      <td style="padding: 8px;">SVM prediction can be slow on large datasets.</td>
    </tr>
    <tr>
      <td style="padding: 8px;"><strong>Best PCA + SVM</strong></td>
      <td style="padding: 8px;">TBD%</td>
    </tr>
    <tr>
      <td style="padding: 8px;"><strong>MLP Classifier</strong></td>
      <td style="padding: 8px;">~99.17%</td>
      <td style="padding: 8px;">Captures complex, non-linear deep patterns.</td>
      <td style="padding: 8px;">Requires scaling, computationally heavy to train (GridSearch).</td>
    </tr>
    <tr>
      <td style="padding: 8px;"><strong>Best MLP Classifier</strong></td>
      <td style="padding: 8px;">~99.02%</td>
    </tr>
  </tbody>
</table>

### Conclusion

Based on the evaluation of the three models (Random Forest, PCA + SVM, and MLP), the **MLP Classifier** achieved the highest overall accuracy at **~99.17%**. 

While all models performed exceptionally well on the dataset (all exceeding 98% accuracy), the MLP's ability to capture complex, non-linear deep patterns made it marginally better for this specific image classification task compared to the others. 

However, if training time and computational resources are a strict constraint, the **Random Forest** model still offers excellent performance (~98.97% when optimized) with much faster training times and more robustness to overfitting. The **PCA + SVM** model is also a solid middle-ground, reducing noise efficiently and performing very well (~99.10%), though predictions can be slower on larger datasets.

Therefore, the **MLP Classifier** is the best model for pure accuracy, while the **Random Forest** might be preferred if rapid training and simplicity are required.